# Correlation-Pruned Fixed-Noise Sparse Bayesian Learning for Passive Intermodulation Cancellation

**Authors:** Oleg Kudryavtsev, Nikita Efimenko, and Anatoliy Efimov.

**Numerical example 5:** Tanh source with a cubic-quintic dictionary and AWGN 35 dB below PIM.

## Scope and reproducibility

This experiment evaluates the combined cubic-quintic canceller for the phase-preserving tanh source observed in circular complex AWGN. The noise power is configured 35 dB below the clean PIM power. Dense ridge-LS and correlation-pruned fixed-noise SBL are fitted to the same noisy observation using the same transmitter-side references.

Observed cancellation is the primary metric because both estimators are fitted directly to $y=p+v$. A simulation-only PIM-component metric is retained solely to separate modeling error from the intentionally added noise in the residual spectrum. All quantities are computed directly from the definitions below.


In [ ]:
from __future__ import annotations

import csv
from dataclasses import dataclass
from itertools import combinations_with_replacement
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.fft import next_fast_len
from scipy.linalg import cholesky, get_lapack_funcs
from scipy.signal import firwin, lfilter

plt.rcParams.update({
    'figure.figsize': (11, 4.8),
    'figure.dpi': 110,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## 1. Clean tanh PIM source

The bounded source is

$$
g_{\tanh}(s)=G\tanh(\gamma|s|)\frac{s}{|s|+\varepsilon},
$$

whose receiver-band output contains cubic, quintic, and higher odd-order components. The generator forms the clean filtered PIM signal and the uncorrupted transmit-side references before any observation noise is added.


In [ ]:
S = 5 * 2048
FS = 256e6
ORDER = 2048
F_SHIFT = 30e6
F_UP = 90e6
NL_GAIN = 1.2 + 3.7j
GAMMA = 2.0
EPS_NL = 1e-12
SEED = 12345

DISCARD = 2 * ORDER + 16
N_VALID = S - DISCARD


@dataclass(frozen=True)
class StandaloneData:
    x_hi_tx: np.ndarray
    x_lo_tx: np.ndarray
    x_hi_at_pim: np.ndarray
    x_lo_at_pim: np.ndarray
    target_filtered: np.ndarray
    lpf: np.ndarray


def nonlinear_transform(sig: np.ndarray, gamma: float = GAMMA) -> np.ndarray:
    """Apply the bounded phase-preserving tanh nonlinearity."""
    mag = np.abs(sig)
    phase = sig / (mag + EPS_NL)
    return NL_GAIN * np.tanh(gamma * mag) * phase


def generate_standalone_data(seed: int = SEED, gamma: float = GAMMA) -> StandaloneData:
    """Generate the two transmit bands, PIM source output, and receiver-filtered target."""
    rng = np.random.RandomState(seed)
    s0 = rng.random_sample(S) + 1j * rng.random_sample(S)

    flt = firwin(ORDER + 1, 0.05, window='hamming', scale=True)
    n_fir = np.arange(ORDER + 1)
    flt_hi = flt * np.exp(1j * (2 * np.pi * F_SHIFT * n_fir / FS))
    flt_lo = flt * np.exp(1j * (2 * np.pi * -F_SHIFT * n_fir / FS))

    sig_hi = lfilter(flt_hi, [1.0], s0)
    sig_lo = lfilter(flt_lo, [1.0], s0)
    sig0 = sig_hi + sig_lo

    flt_in = np.array([
        0.186 - 0.023j, -0.098 + 0.117j, 0.291 - 0.338j, -0.607 + 0.628j,
        1.131 - 1.348j, -1.090 + 0.993j, 1.633 - 1.725j, -1.415 + 1.365j,
        0.760 - 0.668j, -0.256 + 0.211j, 0.045 - 0.037j,
    ])

    sig = lfilter(flt_in, [1.0], sig0)
    sig_nl = nonlinear_transform(sig, gamma)

    n = np.arange(S)
    shifted = sig_nl * np.exp(1j * (2 * np.pi * F_UP * n / FS))
    lpf = firwin(ORDER + 1, 0.2, window='hamming', scale=True)
    target_filtered = lfilter(lpf, [1.0], shifted)

    rot_hi = np.exp(-1j * (2 * np.pi * F_SHIFT * n / FS))
    rot_lo = np.exp(1j * (2 * np.pi * F_SHIFT * n / FS))
    x_hi_tx = sig_hi * rot_hi
    x_lo_tx = sig_lo * rot_lo

    sig_hi_at_pim = lfilter(flt_in, [1.0], sig_hi)
    sig_lo_at_pim = lfilter(flt_in, [1.0], sig_lo)
    x_hi_at_pim = sig_hi_at_pim * rot_hi
    x_lo_at_pim = sig_lo_at_pim * rot_lo

    return StandaloneData(
        x_hi_tx=x_hi_tx,
        x_lo_tx=x_lo_tx,
        x_hi_at_pim=x_hi_at_pim,
        x_lo_at_pim=x_lo_at_pim,
        target_filtered=target_filtered,
        lpf=lpf,
    )


data_clean = generate_standalone_data()
TRAIN = slice(0, N_VALID // 2)
TEST = slice(N_VALID // 2, N_VALID)
print(f'Generated {S} complex samples; {N_VALID} samples remain after transient removal.')


## 2. AWGN observation model

The observed sequence is $y=p+v$, where $p$ is the clean receiver-filtered PIM component and $v$ is zero-mean circular complex AWGN. The configured variance is

$$
\sigma_v^2=\mathbb{E}|p|^2\,10^{-35/10},
$$

evaluated on the valid post-transient samples. The same deterministic noise realization is used for every memory depth and fixed-precision setting. The physical AWGN is distinct from the normalized SBL likelihood precision $\beta$.


In [ ]:
NOISE_SEED = 54321
NOISE_BELOW_PIM_DB = 35.0


def complex_awgn(n: int, variance: float, seed: int) -> np.ndarray:
    """Generate circular complex white Gaussian noise with the requested variance."""
    rng = np.random.default_rng(seed)
    scale = np.sqrt(variance / 2)
    return scale * (rng.standard_normal(n) + 1j * rng.standard_normal(n))


target_clean_full = data_clean.target_filtered.copy()
target_clean_valid = target_clean_full[DISCARD:]
pim_power = float(np.mean(np.abs(target_clean_valid) ** 2))
noise_variance = pim_power * 10 ** (-NOISE_BELOW_PIM_DB / 10)
noise_full = complex_awgn(target_clean_full.size, noise_variance, NOISE_SEED)
noise_valid = noise_full[DISCARD:]

data = StandaloneData(
    x_hi_tx=data_clean.x_hi_tx,
    x_lo_tx=data_clean.x_lo_tx,
    x_hi_at_pim=data_clean.x_hi_at_pim,
    x_lo_at_pim=data_clean.x_lo_at_pim,
    target_filtered=target_clean_full + noise_full,
    lpf=data_clean.lpf,
)

noise_power = float(np.mean(np.abs(noise_valid) ** 2))
empirical_ratio_db = 10 * np.log10(pim_power / noise_power)
print(f'Clean PIM power: {pim_power:.4e}')
print(f'Configured AWGN variance: {noise_variance:.4e}')
print(f'Empirical PIM-to-noise ratio: {empirical_ratio_db:.2f} dB')


## 3. Symmetry-reduced cubic-quintic dictionary

The dictionary combines

$$
x_{\ell,a}x_{\ell,b}x_{h,c}^*,\quad a\le b,
$$

$$
x_{\ell,a}x_{\ell,b}x_{\ell,c}x_{\ell,d}^*x_{h,e}^*,\quad a\le b\le c,
$$

and

$$
x_{\ell,a}x_{\ell,b}x_{h,c}x_{h,d}^*x_{h,e}^*,\quad a\le b,\ d\le e.
$$

For $M$ causal taps per signal, the total candidate count is

$$
N_{3+5}(M)=\binom{M+1}{2}M+\binom{M+2}{3}M^2+\binom{M+1}{2}^2M.
$$

The receiver filter is applied to every generated monomial before estimation.


In [ ]:
WIN_START = DISCARD - ORDER
WIN_LEN = S - WIN_START

FAMILY_NAMES = ('c3', 'q5a', 'q5b')


def _triples(memory: int) -> np.ndarray:
    return np.array(list(combinations_with_replacement(range(memory), 3)), dtype=np.int32)


def _pairs(memory: int) -> np.ndarray:
    return np.array(list(combinations_with_replacement(range(memory), 2)), dtype=np.int32)


@dataclass(frozen=True)
class DictLayout:
    """Describe family offsets and decode global dictionary-column indices."""

    memory: int
    orders: tuple[int, ...]
    pairs: np.ndarray
    triples: np.ndarray
    sizes: tuple[int, ...]
    offsets: tuple[int, ...]

    @property
    def n_cols(self) -> int:
        return int(sum(self.sizes))

    def family_slice(self, name: str) -> slice:
        index = FAMILY_NAMES.index(name)
        return slice(self.offsets[index], self.offsets[index] + self.sizes[index])


def make_layout(memory: int, orders: tuple[int, ...] = (3, 5)) -> DictLayout:
    if memory < 1:
        raise ValueError('memory must be at least one')
    if not orders or any(order not in (3, 5) for order in orders):
        raise ValueError('orders must be a nonempty subset of (3, 5)')

    pairs = _pairs(memory)
    triples = _triples(memory)
    n_pairs, n_triples = len(pairs), len(triples)

    sizes = [
        n_pairs * memory if 3 in orders else 0,
        n_triples * memory * memory if 5 in orders else 0,
        memory * n_pairs * n_pairs if 5 in orders else 0,
    ]
    offsets = [0, sizes[0], sizes[0] + sizes[1]]
    return DictLayout(
        memory=memory,
        orders=tuple(orders),
        pairs=pairs,
        triples=triples,
        sizes=tuple(sizes),
        offsets=tuple(offsets),
    )


def decode(layout: DictLayout, cols: np.ndarray) -> list[tuple[int, np.ndarray, np.ndarray]]:
    """Decode global column indices into family identifiers and lag tuples."""
    cols = np.asarray(cols, dtype=np.int64)
    if cols.size and (cols.min() < 0 or cols.max() >= layout.n_cols):
        raise ValueError('column index is outside the dictionary')

    memory = layout.memory
    n_pairs = len(layout.pairs)
    decoded: list[tuple[int, np.ndarray, np.ndarray]] = []

    for family in range(3):
        size = layout.sizes[family]
        if size == 0:
            continue
        offset = layout.offsets[family]
        local_positions = np.flatnonzero((cols >= offset) & (cols < offset + size))
        if local_positions.size == 0:
            continue
        local_index = cols[local_positions] - offset

        if family == 0:
            pair_index, k3 = np.divmod(local_index, memory)
            lags = np.column_stack([layout.pairs[pair_index], k3])
        elif family == 1:
            triple_index, remainder = np.divmod(local_index, memory * memory)
            k4, k5 = np.divmod(remainder, memory)
            lags = np.column_stack([layout.triples[triple_index], k4, k5])
        else:
            first_pair_index, remainder = np.divmod(local_index, memory * n_pairs)
            k3, second_pair_index = np.divmod(remainder, n_pairs)
            lags = np.column_stack([
                layout.pairs[first_pair_index],
                k3,
                layout.pairs[second_pair_index],
            ])

        decoded.append((family, local_positions, lags.astype(np.int32)))
    return decoded


class BasisBuilder:
    """Build selected filtered dictionary columns without storing all raw monomials."""

    def __init__(
        self,
        data: StandaloneData,
        layout: DictLayout,
        *,
        use_fft: bool = True,
        fft_chunk: int = 128,
    ):
        self.layout = layout
        self.use_fft = use_fft
        self.fft_chunk = max(1, fft_chunk)
        self.lpf = data.lpf

        memory = layout.memory
        rows = np.arange(WIN_START, S)
        self._d_lo = np.stack([data.x_lo_tx[rows - lag] for lag in range(memory)], axis=1)
        self._d_hi = np.stack([data.x_hi_tx[rows - lag] for lag in range(memory)], axis=1)
        self.y = data.target_filtered[WIN_START + ORDER:]
        if self.y.size != N_VALID:
            raise RuntimeError(f'target window has {self.y.size} samples; expected {N_VALID}')

        self._nfft = next_fast_len(WIN_LEN + self.lpf.size - 1)
        self._h_fft = np.fft.fft(self.lpf, self._nfft) if use_fft else None

    @property
    def n_rows(self) -> int:
        return N_VALID

    def _filter(self, block: np.ndarray) -> np.ndarray:
        if not self.use_fft:
            return lfilter(self.lpf, [1.0], block, axis=0)[ORDER:]

        n_cols = block.shape[1]
        out = np.empty((WIN_LEN - ORDER, n_cols), dtype=np.complex128)
        for start in range(0, n_cols, self.fft_chunk):
            selected = slice(start, min(start + self.fft_chunk, n_cols))
            spectrum = np.fft.fft(block[:, selected], self._nfft, axis=0)
            spectrum *= self._h_fft[:, None]
            out[:, selected] = np.fft.ifft(spectrum, axis=0)[ORDER:WIN_LEN]
        return out

    def build(self, cols: np.ndarray, rows: slice | None = None) -> np.ndarray:
        """Return the filtered matrix for the requested columns in the requested order."""
        cols = np.asarray(cols, dtype=np.int64)
        raw = np.empty((WIN_LEN, cols.size), dtype=np.complex128)
        lo, hi = self._d_lo, self._d_hi

        for family, local_positions, lags in decode(self.layout, cols):
            if family == 0:
                k1, k2, k3 = lags.T
                raw[:, local_positions] = lo[:, k1] * lo[:, k2] * np.conj(hi[:, k3])
            elif family == 1:
                k1, k2, k3, k4, k5 = lags.T
                raw[:, local_positions] = (
                    lo[:, k1] * lo[:, k2] * lo[:, k3]
                    * np.conj(lo[:, k4]) * np.conj(hi[:, k5])
                )
            else:
                k1, k2, k3, k4, k5 = lags.T
                raw[:, local_positions] = (
                    lo[:, k1] * lo[:, k2] * hi[:, k3]
                    * np.conj(hi[:, k4]) * np.conj(hi[:, k5])
                )

        filtered = self._filter(raw)
        return filtered if rows is None else filtered[rows]

    def blocks(self, cols: np.ndarray, block: int, rows: slice | None = None):
        """Iterate over filtered dictionary blocks while preserving global column order."""
        cols = np.asarray(cols, dtype=np.int64)
        for start in range(0, cols.size, block):
            selected = slice(start, min(start + block, cols.size))
            yield selected, self.build(cols[selected], rows)


## 4. Candidate-count verification

The code below verifies the pre-screen sizes for the evaluated causal memory depths. These counts quantify the dimensionality reduced by correlation screening and relevance pruning.


In [ ]:
print(f'{"M":>3} {"cubic":>8} {"fifth-A":>10} {"fifth-B":>10} {"total":>10}')
print('-' * 47)
for memory in (2, 4, 6, 8):
    layout = make_layout(memory, orders=(3, 5))
    print(
        f'{memory:>3} {layout.sizes[0]:>8} {layout.sizes[1]:>10} '
        f'{layout.sizes[2]:>10} {layout.n_cols:>10}'
    )


## 5. Dense ridge-LS reference

Ridge-LS fits the full symmetry-reduced dictionary with a relative factor $10^{-4}$ scaled by the mean covariance diagonal. For $P>N_f$, the dual form is used to avoid a $P\times P$ solve without changing the estimator. The coefficients are fitted to the noisy observation, so the irreducible AWGN remains in the observed residual.


In [ ]:
def cancellation_db(target: np.ndarray, residual: np.ndarray) -> float:
    eps = 1e-30
    return 10 * np.log10(
        np.mean(np.abs(target) ** 2) / (np.mean(np.abs(residual) ** 2) + eps)
    )


@dataclass(frozen=True)
class LsResult:
    h: np.ndarray
    pred: np.ndarray
    mode: str


def solve_ls(
    builder: BasisBuilder,
    cols: np.ndarray,
    *,
    train: slice,
    ridge: float = 1e-4,
    block: int = 256,
    mode: str = 'auto',
) -> LsResult:
    """Fit the scale-adaptive ridge-LS estimator in primal or equivalent dual form."""
    cols = np.asarray(cols, dtype=np.int64)
    n_cols = cols.size
    y = builder.y
    y_train = y[train]
    k_train = y_train.size

    if mode == 'auto':
        mode = 'primal' if n_cols <= k_train else 'dual'

    if mode == 'primal':
        x_mat = builder.build(cols)
        x_train = x_mat[train]
        r_mat = x_train.conj().T @ x_train / k_train
        r_vec = x_train.conj().T @ y_train / k_train
        regularization = ridge * np.trace(r_mat).real / n_cols
        h = np.linalg.solve(r_mat + regularization * np.eye(n_cols), r_vec)
        return LsResult(h=h, pred=x_mat @ h, mode=mode)

    if mode != 'dual':
        raise ValueError(f'unknown LS mode: {mode!r}')

    gram = np.zeros((k_train, k_train), dtype=np.complex128)
    trace_accumulator = 0.0
    for _, x_block in builder.blocks(cols, block):
        x_train_block = x_block[train]
        gram += x_train_block @ x_train_block.conj().T
        trace_accumulator += float(np.sum(np.abs(x_train_block) ** 2))

    regularization = ridge * (trace_accumulator / k_train) / n_cols
    gram /= k_train
    gram[np.diag_indices_from(gram)] += regularization
    dual_weights = np.linalg.solve(gram, y_train / k_train)
    del gram

    h = np.empty(n_cols, dtype=np.complex128)
    pred = np.zeros(builder.n_rows, dtype=np.complex128)
    for selected, x_block in builder.blocks(cols, block):
        h_block = x_block[train].conj().T @ dual_weights
        h[selected] = h_block
        pred += x_block @ h_block

    return LsResult(h=h, pred=pred, mode=mode)


## 6. Correlation-pruned fixed-noise SBL

One-shot normalized correlation screening precedes the SBL iterations. The likelihood precision $\beta$ remains fixed within each run, coefficient precisions are updated, and irrelevant columns are removed when their precision becomes non-finite or exceeds $\alpha_{\max}$. The surviving posterior means are rescaled before synthesizing the cancellation replica.

Because the physical AWGN realization is held constant, the external $\beta$ sweep isolates the effect of the inference setting rather than changing the observed signal.


In [ ]:
@dataclass(frozen=True)
class SblConfig:
    beta: float = 100.0
    eta: float = 0.01
    keep_top: int | None = None
    min_keep: int = 1
    alpha_init: float = 1.0
    alpha_max: float = 1e6
    max_iter: int = 500
    tol: float = 1e-6
    eps: float = 1e-12
    linalg: str = 'chol'


@dataclass
class SblState:
    cols: np.ndarray
    alpha: np.ndarray
    mu: np.ndarray
    col_norms: np.ndarray
    iterations: int = 0
    converged: bool = False


def _diag_of_inverse(inv_lower: np.ndarray, chunk: int = 512) -> np.ndarray:
    """Compute diag(L^{-H}L^{-1}) in row blocks to limit temporary memory."""
    diagonal = np.zeros(inv_lower.shape[1])
    for start in range(0, inv_lower.shape[0], chunk):
        block = inv_lower[start:start + chunk]
        diagonal += np.sum(block.real ** 2 + block.imag ** 2, axis=0)
    return diagonal


def _build_system(gram: np.ndarray, alpha: np.ndarray, beta: float) -> np.ndarray:
    system = gram * beta
    system[np.diag_indices_from(system)] += alpha
    return system


def _posterior(
    gram: np.ndarray,
    rhs: np.ndarray,
    alpha: np.ndarray,
    beta: float,
    method: str,
) -> tuple[np.ndarray, np.ndarray]:
    """Return the posterior mean and the diagonal of the posterior covariance."""
    system = _build_system(gram, alpha, beta)
    if method != 'inv':
        try:
            lower = cholesky(system, lower=True, overwrite_a=True, check_finite=False)
            triangular_inverse = get_lapack_funcs('trtri', (lower,))
            inv_lower, info = triangular_inverse(lower, lower=1, overwrite_c=1)
            if info != 0:
                raise np.linalg.LinAlgError(f'trtri returned info={info}')
        except (np.linalg.LinAlgError, ValueError):
            system = _build_system(gram, alpha, beta)
        else:
            sigma_diagonal = _diag_of_inverse(inv_lower)
            mu = beta * (inv_lower.conj().T @ (inv_lower @ rhs))
            return mu, sigma_diagonal

    sigma = np.linalg.inv(system)
    sigma = (sigma + sigma.conj().T) / 2
    return beta * (sigma @ rhs), np.real(np.diag(sigma))


def sbl_core(
    x_normed: np.ndarray,
    y_normed: np.ndarray,
    cfg: SblConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, int, bool]:
    """Run fixed-noise SBL on an already normalized screened matrix."""
    keep = np.arange(x_normed.shape[1])
    x_active = x_normed
    alpha = np.full(x_active.shape[1], cfg.alpha_init, dtype=float)
    previous_mu: np.ndarray | None = None
    mu = np.zeros(x_active.shape[1], dtype=complex)
    converged = False
    iteration = 0

    for iteration in range(1, cfg.max_iter + 1):
        gram = x_active.conj().T @ x_active
        rhs = x_active.conj().T @ y_normed
        mu, sigma_diagonal = _posterior(gram, rhs, alpha, cfg.beta, cfg.linalg)

        gamma_eff = np.clip(1.0 - alpha * sigma_diagonal, cfg.eps, 1.0)
        alpha_new = np.clip(
            gamma_eff / (np.abs(mu) ** 2 + cfg.eps),
            cfg.eps,
            1e16,
        )

        survive = np.isfinite(alpha_new) & (alpha_new <= cfg.alpha_max)
        if survive.sum() == 0:
            survive[np.argmin(alpha_new)] = True

        if survive.sum() < survive.size:
            x_active = x_active[:, survive]
            keep = keep[survive]
            alpha = alpha_new[survive]
            mu = mu[survive]
            previous_mu = None
            continue

        if previous_mu is not None:
            denominator = max(np.linalg.norm(previous_mu), cfg.eps)
            relative_change = np.linalg.norm(mu - previous_mu) / denominator
            if relative_change < cfg.tol:
                alpha = alpha_new
                converged = True
                break

        previous_mu = mu.copy()
        alpha = alpha_new

    return keep, alpha, mu, iteration, converged


def _screen(scores: np.ndarray, cfg: SblConfig) -> np.ndarray:
    keep = scores > cfg.eta
    if cfg.keep_top is not None:
        keep[np.argsort(scores)[-min(cfg.keep_top, scores.size):]] = True
    if keep.sum() < cfg.min_keep:
        keep[:] = False
        keep[np.argsort(scores)[-min(cfg.min_keep, scores.size):]] = True
    return np.flatnonzero(keep)


def solve_sbl_full(
    builder: BasisBuilder,
    cols: np.ndarray,
    cfg: SblConfig,
    *,
    train: slice,
) -> tuple[SblState, int]:
    """Apply one-shot normalized correlation screening followed by fixed-noise SBL."""
    cols = np.asarray(cols, dtype=np.int64)
    y_train = builder.y[train]
    y_norm = float(np.linalg.norm(y_train))
    if y_norm <= cfg.eps:
        raise ValueError('the fitting target is numerically zero')

    x_train = builder.build(cols, train)
    col_norms = np.linalg.norm(x_train, axis=0)
    nonzero = np.flatnonzero(col_norms > cfg.eps)
    if nonzero.size == 0:
        raise ValueError('all dictionary columns are numerically zero')
    if nonzero.size < cols.size:
        x_train = x_train[:, nonzero]

    x_train /= col_norms[nonzero]
    y_unit = y_train / y_norm
    scores = np.abs(x_train.conj().T @ y_unit)
    screened = _screen(scores, cfg)
    x_screened = x_train[:, screened]

    keep, alpha, mu, iterations, converged = sbl_core(x_screened, y_unit, cfg)
    selected_nonzero = nonzero[screened[keep]]
    state = SblState(
        cols=cols[selected_nonzero],
        alpha=alpha,
        mu=mu,
        col_norms=col_norms[selected_nonzero],
        iterations=iterations,
        converged=converged,
    )
    return state, int(screened.size)


def sbl_weights(state: SblState, y_norm: float) -> np.ndarray:
    """Rescale posterior means to the original target and dictionary-column norms."""
    return (y_norm / state.col_norms) * state.mu


## 7. Noisy cubic-quintic experiment

The complete sweep uses $M\in\{2,4,6,8\}$ and $\beta\in\{10^2,10^4,10^6\}$ with $\alpha_{\max}=10^6$ and at most 500 SBL iterations. The main table reports cancellation on the actually observed signal $y=p+v$, active support size, iteration count, and convergence. PIM-only metrics are stored in the CSV for simulation diagnostics.


In [ ]:
OUT = Path('out') / 'quintic_noisy'
OUT.mkdir(parents=True, exist_ok=True)

MEMORIES = (2, 4, 6, 8)
BETAS = (1e2, 1e4, 1e6)

ls_cache: dict[int, dict] = {}
for memory in MEMORIES:
    layout = make_layout(memory, orders=(3, 5))
    builder = BasisBuilder(data, layout)
    all_cols = np.arange(layout.n_cols)
    ls_result = solve_ls(builder, all_cols, train=TRAIN, block=192)
    observed = builder.y
    ls_cache[memory] = {
        'layout': layout,
        'builder': builder,
        'pred': ls_result.pred,
        'observed_test_db': cancellation_db(
            observed[TEST],
            observed[TEST] - ls_result.pred[TEST],
        ),
        'pim_test_db': cancellation_db(
            target_clean_valid[TEST],
            target_clean_valid[TEST] - ls_result.pred[TEST],
        ),
        'mode': ls_result.mode,
    }

rows: list[dict] = []
series: dict[tuple[float, int], tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]] = {}

for beta in BETAS:
    for memory in MEMORIES:
        cached = ls_cache[memory]
        builder = cached['builder']
        layout = cached['layout']
        observed = builder.y
        cfg = SblConfig(
            beta=beta,
            eta=0.01,
            keep_top=None,
            min_keep=1,
            alpha_max=1e6,
            max_iter=500,
            tol=1e-6,
            linalg='chol',
        )
        state, screened = solve_sbl_full(
            builder,
            np.arange(layout.n_cols),
            cfg,
            train=TRAIN,
        )
        pred_sbl = builder.build(state.cols) @ sbl_weights(
            state,
            float(np.linalg.norm(observed[TRAIN])),
        )
        row = {
            'beta': beta,
            'memory': memory,
            'n_cols': layout.n_cols,
            'ls_observed_db': cached['observed_test_db'],
            'sbl_observed_db': cancellation_db(
                observed[TEST],
                observed[TEST] - pred_sbl[TEST],
            ),
            'ls_pim_db': cached['pim_test_db'],
            'sbl_pim_db': cancellation_db(
                target_clean_valid[TEST],
                target_clean_valid[TEST] - pred_sbl[TEST],
            ),
            'screened': screened,
            'active': int(state.cols.size),
            'iterations': state.iterations,
            'converged': state.converged,
            'ls_mode': cached['mode'],
        }
        rows.append(row)
        series[(beta, memory)] = (
            observed,
            target_clean_valid,
            noise_valid,
            cached['pred'],
            pred_sbl,
        )

print(
    f'{"beta":>9} {"M":>3} {"LS observed":>12} {"SBL observed":>13} '
    f'{"active":>8} {"iter.":>7} {"conv.":>7}'
)
print('-' * 68)
for row in rows:
    print(
        f'{row["beta"]:>9.0g} {row["memory"]:>3} '
        f'{row["ls_observed_db"]:>12.1f} {row["sbl_observed_db"]:>13.1f} '
        f'{row["active"]:>8} {row["iterations"]:>7} {str(row["converged"]):>7}'
    )

csv_path = OUT / 'tanh_cubic_quintic_awgn_results.csv'
with csv_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(f'Results saved to {csv_path}')


## 8. Precision sensitivity and noisy residual spectrum

The sensitivity plot uses observed test cancellation. The selected spectrum uses $M=6$ and $\beta=10^6$ and displays the noisy observation, clean PIM, added AWGN, ridge-LS residual, SBL residual, and SBL PIM-only error. The observed residual cannot fall substantially below the added-noise floor even when the PIM model is accurate.


In [ ]:
def spectrum_db(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    freqs = np.fft.fftshift(np.fft.fftfreq(x.size, 1.0 / FS)) / 1e6
    magnitude = 20 * np.log10(np.abs(np.fft.fftshift(np.fft.fft(x))) + 1e-300)
    return freqs, magnitude


fig, ax = plt.subplots(figsize=(8.5, 4.8))
for memory in MEMORIES:
    x_values = [row['beta'] for row in rows if row['memory'] == memory]
    y_values = [row['sbl_observed_db'] for row in rows if row['memory'] == memory]
    ax.semilogx(x_values, y_values, 'o-', label=f'SBL M={memory}')
best_ls = max(rows, key=lambda row: row['ls_observed_db'])
ax.axhline(
    best_ls['ls_observed_db'],
    ls=':',
    lw=1.0,
    label=(
        f'Best ridge-LS: M={best_ls["memory"]}, '
        f'{best_ls["ls_observed_db"]:.1f} dB observed'
    ),
)
ax.set_xlabel('Fixed likelihood precision beta')
ax.set_ylabel('Observed test cancellation, dB')
ax.set_title('Noisy tanh source: cubic-quintic SBL sensitivity to fixed precision')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'beta_sensitivity_tanh_cubic_quintic_awgn.png', dpi=130)
plt.show()

selected_beta = 1e6
selected_memory = 6
selected = next(
    row for row in rows
    if row['beta'] == selected_beta and row['memory'] == selected_memory
)
observed, clean_pim, noise, pred_ls, pred_sbl = series[(selected_beta, selected_memory)]

freqs, observed_db = spectrum_db(observed)
_, clean_db = spectrum_db(clean_pim)
_, noise_db = spectrum_db(noise)
_, ls_residual_db = spectrum_db(observed - pred_ls)
_, sbl_residual_db = spectrum_db(observed - pred_sbl)
_, sbl_pim_error_db = spectrum_db(clean_pim - pred_sbl)

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(freqs, observed_db, lw=1.0, label='Observed signal: clean PIM + AWGN')
ax.plot(freqs, clean_db, lw=1.0, label='Clean PIM reference')
ax.plot(freqs, noise_db, lw=0.9, ls='--', label='Added AWGN only')
ax.plot(
    freqs,
    ls_residual_db,
    lw=1.0,
    label=(
        f'Ridge-LS observed residual, M={selected_memory}, '
        f'{selected["ls_observed_db"]:.1f} dB'
    ),
)
ax.plot(
    freqs,
    sbl_residual_db,
    lw=1.0,
    label=(
        f'SBL observed residual, M={selected_memory}, beta={selected_beta:g}, '
        f'active={selected["active"]}, {selected["sbl_observed_db"]:.1f} dB'
    ),
)
ax.plot(
    freqs,
    sbl_pim_error_db,
    lw=1.0,
    ls=':',
    label=f'SBL PIM-only error, {selected["sbl_pim_db"]:.1f} dB',
)
ax.set_xlim(-28, 28)
ax.set_xlabel('Frequency, MHz')
ax.set_ylabel('20*log10|FFT(.)|, dB')
ax.set_title('Tanh source with cubic-quintic cancellation and AWGN 35 dB below PIM')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'spectrum_tanh_cubic_quintic_awgn.png', dpi=130)
plt.show()


## Interpretation of the selected operating point

With the deterministic configuration defined above, at $M=6$ and $\beta=10^6$, dense ridge-LS and SBL both reach approximately $28.8$ dB cancellation on the noisy observation. SBL retains 22 of the 4,788 pre-screen candidates. The corresponding PIM-only diagnostic is approximately $29.9$ dB, showing that the remaining difference in the observed metric is dominated by the intentionally added AWGN rather than by a loss of higher-order modeling capability.
